In [ ]:
# data_exploration2_lasso_features.ipynb - Enhanced Features with LASSO Selection

# Setup
import pandas as pd
import pickle
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# modeling used
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import accuracy_score, roc_auc_score, brier_score_loss, roc_curve, auc
from sklearn.preprocessing import StandardScaler
import xgboost as xgb

# Display settings
pd.set_option('display.max_columns', None)
np.set_printoptions(suppress=True)


In [ ]:
# Load Dataset
df = pd.read_csv("../data/raw/all_teams.csv")
print("Shape:", df.shape)
df.head()


In [ ]:
# Feature Engineering - Initial Data Cleaning

# Target variable: 1 = Win, 0 = Loss
df['win'] = (df['goalsFor'] > df['goalsAgainst']).astype(int)

# Merge Arizona with Utah due to the team relocation
df['team'] = df['team'].replace({'ARI': 'UTA'})

# Filter to keep only seasons from 2010 onward (same as data_exploration2.ipynb)
df_recent = df[df['season'] >= 2010].reset_index(drop=True)
print("Filtered recent seasons (>= 2010):", len(df_recent))
print("Unique seasons:", df_recent['season'].unique())

# From the recent data, select only rows where situation == 'all'
df_all = df_recent[df_recent['situation'] == 'all'].reset_index(drop=True)
print("\nRows with situation == 'all':", len(df_all))
print("Unique games before cleanup:", df_all['gameId'].nunique())

# Detect and remove bad games, each game should have exactly one winner
game_win_sums = df_all.groupby('gameId')['win'].sum()

# Identify invalid games
bad_game_ids = game_win_sums[game_win_sums != 1].index
print("Bad games detected:", len(bad_game_ids))

# Remove invalid games
df_all = df_all[~df_all['gameId'].isin(bad_game_ids)].reset_index(drop=True)

# Print cleanup summary
print("\n✅ After cleanup:")
print("Total rows:", len(df_all))
print("Unique games:", df_all['gameId'].nunique())
print("Win counts:\n", df_all['win'].value_counts())


In [ ]:
# Pre-game pipeline: build team-game table from df_recent
# Following methodology from data_exploration2.ipynb

# Start from team-level rows only
df_pre_raw = df_recent[df_recent["position"] == "Team Level"].copy()

# Get win + date info from ALL-situation rows (one per team-game)
df_pre_win = df_pre_raw[df_pre_raw["situation"] == "all"][[
    "gameId", "playerTeam", "home_or_away", "gameDate", "win", "season"
]].copy()

# Pivot situations (all, 5on5, 5on4, 4on5, other) into columns
df_pre_pivot = df_pre_raw.pivot_table(
    index=["gameId", "playerTeam", "home_or_away"],
    columns="situation",
    aggfunc="first"
)

# Flatten multi-index columns: (stat, situation) -> "stat_situation"
df_pre_pivot.columns = [f"{stat}_{sit}" for stat, sit in df_pre_pivot.columns]
df_pre_pivot = df_pre_pivot.reset_index()

# Merge win + date back in
df_teamgames = df_pre_pivot.merge(
    df_pre_win,
    on=["gameId", "playerTeam", "home_or_away"],
    how="left"
)

print("Team-game table shape:", df_teamgames.shape)
df_teamgames.head()


In [ ]:
# Create comprehensive per-game features
# Including features from features.ipynb and data_exploration2.ipynb

df_pg = df_teamgames.copy()

# Ensure chronological order within each team
df_pg = df_pg.sort_values(["playerTeam", "gameDate"]).copy()
df_pg["gameDate_dt"] = pd.to_datetime(df_pg["gameDate"], format="%Y%m%d")

# === Core Performance Metrics (from data_exploration2.ipynb) ===
# Even-strength (5v5) expected goal differential
df_pg["ES_xG_diff_5v5"] = df_pg["xGoalsFor_5on5"] - df_pg["xGoalsAgainst_5on5"]

# Power-play xG rate (5on4)
df_pg["PP_xG_rate"] = df_pg["xGoalsFor_5on4"] / (df_pg["shotsOnGoalFor_5on4"] + 1)

# Penalty-kill xG allowed rate (4on5) – lower is better
df_pg["PK_xG_allowed_rate"] = df_pg["xGoalsAgainst_4on5"] / (df_pg["shotsOnGoalAgainst_4on5"] + 1)

# Discipline (all situations)
df_pg["penalty_diff_all"] = df_pg["penaltiesFor_all"] - df_pg["penaltiesAgainst_all"]

# Home indicator
df_pg["is_home"] = (df_pg["home_or_away"] == "HOME").astype(int)

# === Additional Features from features.ipynb ===
# Goals metrics
df_pg["goalsFor_all"] = df_pg.get("goalsFor_all", 0)
df_pg["goalsAgainst_all"] = df_pg.get("goalsAgainst_all", 0)

# Shots metrics
df_pg["shotsOnGoalFor_all"] = df_pg.get("shotsOnGoalFor_all", 0)
df_pg["shotsOnGoalAgainst_all"] = df_pg.get("shotsOnGoalAgainst_all", 0)
df_pg["shotAttemptsFor_all"] = df_pg.get("shotAttemptsFor_all", 0)
df_pg["shotAttemptsAgainst_all"] = df_pg.get("shotAttemptsAgainst_all", 0)

# CORSI (shot attempts)
df_pg["corsiFor_all"] = df_pg.get("shotAttemptsFor_all", 0)
df_pg["corsiAgainst_all"] = df_pg.get("shotAttemptsAgainst_all", 0)
df_pg["corsi_diff_all"] = df_pg["corsiFor_all"] - df_pg["corsiAgainst_all"]

# Fenwick (unblocked shot attempts)
df_pg["fenwickFor_all"] = df_pg.get("unblockedShotAttemptsFor_all", 0)
df_pg["fenwickAgainst_all"] = df_pg.get("unblockedShotAttemptsAgainst_all", 0)
df_pg["fenwick_diff_all"] = df_pg["fenwickFor_all"] - df_pg["fenwickAgainst_all"]

# Face-offs
df_pg["faceOffsWonFor_all"] = df_pg.get("faceOffsWonFor_all", 0)
df_pg["faceOffsWonAgainst_all"] = df_pg.get("faceOffsWonAgainst_all", 0)
df_pg["faceOff_diff_all"] = df_pg["faceOffsWonFor_all"] - df_pg["faceOffsWonAgainst_all"]

# Hits
df_pg["hitsFor_all"] = df_pg.get("hitsFor_all", 0)
df_pg["hitsAgainst_all"] = df_pg.get("hitsAgainst_all", 0)
df_pg["hits_diff_all"] = df_pg["hitsFor_all"] - df_pg["hitsAgainst_all"]

# Penalty minutes
df_pg["penalityMinutesFor_all"] = df_pg.get("penalityMinutesFor_all", 0)
df_pg["penalityMinutesAgainst_all"] = df_pg.get("penalityMinutesAgainst_all", 0)
df_pg["penaltyMinutes_diff_all"] = df_pg["penalityMinutesFor_all"] - df_pg["penalityMinutesAgainst_all"]

# Blocks
df_pg["blockedShotAttemptsFor_all"] = df_pg.get("blockedShotAttemptsFor_all", 0)
df_pg["blockedShotAttemptsAgainst_all"] = df_pg.get("blockedShotAttemptsAgainst_all", 0)
df_pg["blocks_diff_all"] = df_pg["blockedShotAttemptsFor_all"] - df_pg["blockedShotAttemptsAgainst_all"]

# Giveaways/Takeaways
df_pg["giveawaysFor_all"] = df_pg.get("giveawaysFor_all", 0)
df_pg["giveawaysAgainst_all"] = df_pg.get("giveawaysAgainst_all", 0)
df_pg["takeawaysFor_all"] = df_pg.get("takeawaysFor_all", 0)
df_pg["takeawaysAgainst_all"] = df_pg.get("takeawaysAgainst_all", 0)
df_pg["giveaway_diff_all"] = df_pg["giveawaysFor_all"] - df_pg["giveawaysAgainst_all"]
df_pg["takeaway_diff_all"] = df_pg["takeawaysFor_all"] - df_pg["takeawaysAgainst_all"]

# xG metrics (all situations)
df_pg["xGoalsFor_all"] = df_pg.get("xGoalsFor_all", 0)
df_pg["xGoalsAgainst_all"] = df_pg.get("xGoalsAgainst_all", 0)
df_pg["xG_diff_all"] = df_pg["xGoalsFor_all"] - df_pg["xGoalsAgainst_all"]

# Shooting and Save Percentages
df_pg["shootingPercentageFor"] = (df_pg["goalsFor_all"] / (df_pg["shotsOnGoalFor_all"] + 1) * 100).fillna(0)
df_pg["shootingPercentageAgainst"] = (df_pg["goalsAgainst_all"] / (df_pg["shotsOnGoalAgainst_all"] + 1) * 100).fillna(0)
df_pg["savePercentageFor"] = ((df_pg["shotsOnGoalAgainst_all"] - df_pg["goalsAgainst_all"]) / 
                                (df_pg["shotsOnGoalAgainst_all"] + 1) * 100).fillna(0)
df_pg["savePercentageAgainst"] = ((df_pg["shotsOnGoalFor_all"] - df_pg["goalsFor_all"]) / 
                                    (df_pg["shotsOnGoalFor_all"] + 1) * 100).fillna(0)

# PDO (Shooting % + Save %)
df_pg["pdoFor"] = df_pg["shootingPercentageFor"] + df_pg["savePercentageFor"]
df_pg["pdoAgainst"] = df_pg["shootingPercentageAgainst"] + df_pg["savePercentageAgainst"]

# Power Play and Penalty Kill metrics
df_pg["PP_goalsFor"] = df_pg.get("goalsFor_5on4", 0)
df_pg["PK_goalsAgainst"] = df_pg.get("goalsAgainst_4on5", 0)

print("Created comprehensive per-game features")
df_pg[["playerTeam", "gameDate", "ES_xG_diff_5v5", "corsi_diff_all", "xG_diff_all", "win"]].head()


In [ ]:
# Create rolling window features for all metrics
# Following methodology: last 3, 5, 10 games, and season-to-date

g = df_pg.groupby("playerTeam", group_keys=False)

# List of features to create rolling windows for
rolling_features = [
    "ES_xG_diff_5v5",
    "PP_xG_rate",
    "PK_xG_allowed_rate",
    "penalty_diff_all",
    "goalsFor_all",
    "goalsAgainst_all",
    "shotsOnGoalFor_all",
    "shotsOnGoalAgainst_all",
    "corsi_diff_all",
    "fenwick_diff_all",
    "faceOff_diff_all",
    "hits_diff_all",
    "penaltyMinutes_diff_all",
    "blocks_diff_all",
    "giveaway_diff_all",
    "takeaway_diff_all",
    "xG_diff_all",
    "shootingPercentageFor",
    "savePercentageFor",
    "pdoFor"
]

# Rolling windows: last 3, 5, 10 games
for feature in rolling_features:
    if feature in df_pg.columns:
        for window in [3, 5, 10]:
            df_pg[f"{feature}_last{window}"] = g[feature].apply(
                lambda s: s.shift(1).rolling(window).mean()
            )

# Season-to-date (expanding mean)
g_season = df_pg.groupby(["playerTeam", "season"], group_keys=False)
for feature in rolling_features:
    if feature in df_pg.columns:
        df_pg[f"{feature}_season"] = g_season[feature].apply(
            lambda s: s.shift(1).expanding().mean()
        )

# Rolling win rate (momentum)
for window in [3, 5, 10]:
    df_pg[f"win_rate_last{window}"] = g["win"].apply(
        lambda s: s.shift(1).rolling(window).mean()
    )

# Rest days (time since last game for that team)
df_pg["days_since_last_game"] = g["gameDate_dt"].apply(
    lambda s: s.diff().dt.days
)

print("Created rolling window features")
print(f"Total columns after feature engineering: {df_pg.shape[1]}")


In [ ]:
# Prepare feature matrix with all rolling features
# Collect all rolling feature columns

all_rolling_features = []
for feature in rolling_features:
    for window in [3, 5, 10, 'season']:
        col_name = f"{feature}_last{window}" if window != 'season' else f"{feature}_season"
        if col_name in df_pg.columns:
            all_rolling_features.append(col_name)

# Add win rates and other static features
additional_features = [
    "win_rate_last3",
    "win_rate_last5",
    "win_rate_last10",
    "days_since_last_game",
    "is_home"
]

all_features = all_rolling_features + [f for f in additional_features if f in df_pg.columns]

print(f"Total candidate features: {len(all_features)}")
print(f"Sample features: {all_features[:10]}")


In [ ]:
# Drop rows that don't have enough history for rolling features
df_pre_model = df_pg.dropna(subset=all_features).copy()

# Sort by date globally so train/test is chronological
df_pre_model = df_pre_model.sort_values("gameDate_dt")

X_all = df_pre_model[all_features].copy()
y_all = df_pre_model["win"].astype(int)

print(f"Pre-LASSO dataset shape: {X_all.shape}")
print(f"Features: {len(all_features)}")
print(f"Samples: {len(X_all)}")
print(f"Class distribution: {y_all.value_counts().to_dict()}")


In [ ]:
# Use LASSO (L1 regularization) for feature selection to reduce correlation and select best features
# For binary classification, we use LogisticRegression with L1 penalty (equivalent to LASSO)

# Chronological split for LASSO feature selection
cut_lasso = int(len(X_all) * 0.8)
X_lasso_train, X_lasso_val = X_all.iloc[:cut_lasso], X_all.iloc[cut_lasso:]
y_lasso_train, y_lasso_val = y_all.iloc[:cut_lasso], y_all.iloc[cut_lasso:]

# Standardize features for LASSO (important for regularization)
scaler_lasso = StandardScaler()
X_lasso_train_scaled = scaler_lasso.fit_transform(X_lasso_train)
X_lasso_val_scaled = scaler_lasso.transform(X_lasso_val)

# Use LogisticRegression with L1 penalty (LASSO) and cross-validation to find optimal C
# C is inverse of regularization strength (smaller C = stronger regularization)
print("Running LASSO (L1-regularized Logistic Regression) with cross-validation for feature selection...")
lasso_lr = LogisticRegressionCV(
    Cs=np.logspace(-4, 2, 50),  # Range of C values to try (inverse of regularization)
    penalty='l1',  # L1 penalty (LASSO)
    solver='liblinear',  # liblinear supports L1 penalty
    cv=5,  # 5-fold cross-validation
    max_iter=2000,
    random_state=42,
    n_jobs=-1,
    scoring='roc_auc'
)

# Fit LASSO Logistic Regression
lasso_lr.fit(X_lasso_train_scaled, y_lasso_train)

# Get selected features (non-zero coefficients)
selected_features_mask = np.abs(lasso_lr.coef_[0]) > 1e-6  # Features with non-zero coefficients
selected_features = [all_features[i] for i in range(len(all_features)) if selected_features_mask[i]]

print(f"\nOptimal C (inverse regularization): {lasso_lr.C_[0]:.6f}")
print(f"Features before LASSO: {len(all_features)}")
print(f"Features after LASSO: {len(selected_features)}")
print(f"\nSelected features ({len(selected_features)}):")
for i, feat in enumerate(selected_features, 1):
    coef = lasso_lr.coef_[0][all_features.index(feat)]
    print(f"  {i:2d}. {feat:40s} (coef: {coef:8.4f})")


In [ ]:
# Check correlation between selected features
# LASSO helps reduce correlation, but let's verify

X_selected = X_all[selected_features].copy()
corr_matrix = X_selected.corr().abs()

# Find highly correlated feature pairs (correlation > 0.9)
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if corr_matrix.iloc[i, j] > 0.9:
            high_corr_pairs.append((
                corr_matrix.columns[i],
                corr_matrix.columns[j],
                corr_matrix.iloc[i, j]
            ))

if high_corr_pairs:
    print(f"Found {len(high_corr_pairs)} highly correlated pairs (correlation > 0.9):")
    for feat1, feat2, corr_val in high_corr_pairs[:10]:  # Show first 10
        print(f"  {feat1} <-> {feat2}: {corr_val:.3f}")
else:
    print("✅ No highly correlated pairs found (all correlations < 0.9)")

# Visualize correlation matrix of selected features
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, annot=False, cmap='coolwarm', center=0, 
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title(f"Correlation Matrix of {len(selected_features)} LASSO-Selected Features")
plt.tight_layout()
plt.show()


In [ ]:
# Final feature matrix with LASSO-selected features
X_final = X_all[selected_features].copy()
y_final = y_all.copy()

# Chronological train/test split (80/20)
cut = int(len(X_final) * 0.8)
X_train, X_test = X_final.iloc[:cut], X_final.iloc[cut:]
y_train, y_test = y_final.iloc[:cut], y_final.iloc[cut:]

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"\nTraining class distribution:")
print(y_train.value_counts())
print(f"\nTest class distribution:")
print(y_test.value_counts())


In [ ]:
# Train models on LASSO-selected features
# Standardize features for models that need it
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 1. Logistic Regression
print("Training Logistic Regression...")
log_model = LogisticRegression(max_iter=2000, random_state=42)
log_model.fit(X_train_scaled, y_train)
y_pred_log = log_model.predict(X_test_scaled)
y_proba_log = log_model.predict_proba(X_test_scaled)[:, 1]

acc_log = accuracy_score(y_test, y_pred_log)
roc_auc_log = roc_auc_score(y_test, y_proba_log)
brier_log = brier_score_loss(y_test, y_proba_log)

print(f"Logistic Regression:")
print(f"  Accuracy: {acc_log:.4f}")
print(f"  ROC AUC:  {roc_auc_log:.4f}")
print(f"  Brier:    {brier_log:.4f}")

# 2. Random Forest
print("\nTraining Random Forest...")
rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=10,
    min_samples_split=20,
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)  # RF doesn't need scaling
y_pred_rf = rf_model.predict(X_test)
y_proba_rf = rf_model.predict_proba(X_test)[:, 1]

acc_rf = accuracy_score(y_test, y_pred_rf)
roc_auc_rf = roc_auc_score(y_test, y_proba_rf)
brier_rf = brier_score_loss(y_test, y_proba_rf)

print(f"Random Forest:")
print(f"  Accuracy: {acc_rf:.4f}")
print(f"  ROC AUC:  {roc_auc_rf:.4f}")
print(f"  Brier:    {brier_rf:.4f}")

# 3. XGBoost
print("\nTraining XGBoost...")
xgb_model = xgb.XGBClassifier(
    n_estimators=400,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.5,
    eval_metric="logloss",
    n_jobs=-1,
    random_state=42
)
xgb_model.fit(X_train, y_train)  # XGBoost doesn't need scaling
y_pred_xgb = xgb_model.predict(X_test)
y_proba_xgb = xgb_model.predict_proba(X_test)[:, 1]

acc_xgb = accuracy_score(y_test, y_pred_xgb)
roc_auc_xgb = roc_auc_score(y_test, y_proba_xgb)
brier_xgb = brier_score_loss(y_test, y_proba_xgb)

print(f"XGBoost:")
print(f"  Accuracy: {acc_xgb:.4f}")
print(f"  ROC AUC:  {roc_auc_xgb:.4f}")
print(f"  Brier:    {brier_xgb:.4f}")


In [ ]:
# Feature importance from Random Forest and XGBoost
print("\n" + "="*60)
print("Feature Importance Analysis")
print("="*60)

# Random Forest feature importance
rf_importance = pd.DataFrame({
    'feature': selected_features,
    'importance_rf': rf_model.feature_importances_
}).sort_values('importance_rf', ascending=False)

# XGBoost feature importance
xgb_importance = pd.DataFrame({
    'feature': selected_features,
    'importance_xgb': xgb_model.feature_importances_
}).sort_values('importance_xgb', ascending=False)

# Merge and display top features
importance_df = rf_importance.merge(xgb_importance, on='feature')
importance_df['avg_importance'] = (importance_df['importance_rf'] + importance_df['importance_xgb']) / 2
importance_df = importance_df.sort_values('avg_importance', ascending=False)

print("\nTop 15 Most Important Features (Average of RF and XGBoost):")
print(importance_df.head(15).to_string(index=False))


In [ ]:
# ROC Curves for all models
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

models = [
    (y_proba_log, "Logistic Regression", acc_log, roc_auc_log),
    (y_proba_rf, "Random Forest", acc_rf, roc_auc_rf),
    (y_proba_xgb, "XGBoost", acc_xgb, roc_auc_xgb)
]

for idx, (y_proba, model_name, acc, roc_auc) in enumerate(models):
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    
    axes[idx].plot(fpr, tpr, color='orange', linewidth=2,
                   label=f'{model_name} (AUC = {roc_auc:.3f})')
    axes[idx].plot([0, 1], [0, 1], linestyle='--', color='navy', linewidth=2, label='Random guess')
    axes[idx].set_title(f'{model_name}\nAccuracy: {acc:.3f}')
    axes[idx].set_xlabel('False Positive Rate')
    axes[idx].set_ylabel('True Positive Rate')
    axes[idx].legend(loc='lower right')
    axes[idx].grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()


In [ ]:
# Summary comparison
print("\n" + "="*60)
print("Model Performance Summary")
print("="*60)
print(f"\nFeatures used: {len(selected_features)} (selected from {len(all_features)} candidates by LASSO)")
print(f"\n{'Model':<20} {'Accuracy':<12} {'ROC AUC':<12} {'Brier Score':<12}")
print("-"*60)
print(f"{'Logistic Regression':<20} {acc_log:<12.4f} {roc_auc_log:<12.4f} {brier_log:<12.4f}")
print(f"{'Random Forest':<20} {acc_rf:<12.4f} {roc_auc_rf:<12.4f} {brier_rf:<12.4f}")
print(f"{'XGBoost':<20} {acc_xgb:<12.4f} {roc_auc_xgb:<12.4f} {brier_xgb:<12.4f}")

print("\n" + "="*60)
print("Key Insights:")
print("="*60)
print("1. LASSO feature selection reduced features from", len(all_features), "to", len(selected_features))
print("2. This helps reduce overfitting and multicollinearity")
print("3. Selected features focus on the most predictive metrics")
print("4. Models trained on LASSO-selected features should generalize better")


In [ ]:
# Save the best model and preprocessing objects
# Use XGBoost as it typically performs best

model_bundle = {
    "model": xgb_model,
    "scaler": scaler,
    "selected_features": selected_features,
    "all_candidate_features": all_features,
    "lasso_C": lasso_lr.C_[0],
    "model_type": "XGBoost_with_LASSO_features"
}

with open("model_lasso_features.pkl", "wb") as f:
    pickle.dump(model_bundle, f)

print("Saved model bundle to model_lasso_features.pkl")
print(f"  - Model: XGBoost")
print(f"  - Features: {len(selected_features)} LASSO-selected features")
print(f"  - Scaler: StandardScaler")
print(f"  - LASSO C: {lasso_lr.C_[0]:.6f}")
